# Self-Editing Python Agent — LangChain & LangGraph Core Components

**Goal:** Build a small coding agent that can inspect a Python file, edit it, run it, and report the result.


## Architecture

1. **Chat model** — understands the request and decides what to do.
2. **Tools** — Python functions the model can call (`read_file`, `write_file`, `run_code`, math tools).
3. **System prompt** — defines the agent's role and operating rules.
4. **Agent** — connects the model, tools, and prompt into a tool-calling loop.
5. **InMemorySaver** — stores conversation/checkpoint state in RAM for the configured thread.
6. **Invoke** — sends the user's request and receives the final agent result.

```text
User request
    ↓
create_agent
    ├── System prompt (rules)
    ├── Chat model (reasoning + tool selection)
    ├── Tools (read / write / run )
    └── InMemorySaver (thread checkpoint)
    ↓
Agent response
```


## 1. Install dependencies

Run this once in your notebook environment. Package versions and import paths can change, so if an import fails, check the current LangChain/LangGraph documentation for your installed version.

In [10]:
%pip install -U langchain langgraph langchain-groq python-dotenv

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


## 2. Imports and environment

- `os`: read environment variables.
- `load_dotenv`: loads values from a local `.env` file.
- `Path`: safer path handling than manually joining strings.
- `subprocess`: runs a Python script as a separate process.
- `sys`: gets the current Python interpreter path and command-line args.
- `tool`: turns a Python function into a model-callable tool.
- `init_chat_model`: creates a chat model using a provider.
- `create_agent`: builds the tool-calling agent.
- `InMemorySaver`: keeps checkpoint state in memory (not persistent after process restart).


In [1]:
import os
import sys
import subprocess
from pathlib import Path

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")
if not api_key:
    raise ValueError("GROQ_API_KEY not found. Add it to your .env file.")


### 3. Tools
### What is a Tool?
A tool is a **Python function** that a model can use to perform a specific task.

The model decides when to call a tool, and the function performs the actual task. The tool's description or docstring explains what the tool does.
### `read_file`
The read_file function reads and returns the contents of a file. It checks whether the file exists and rejects directories or invalid file paths.


In [2]:
@tool
def read_file(file_input: str) -> str:
    """Read and return the contents of an existing text file. Pass an exact file path."""
    file_path = Path(file_input)

    if not file_path.exists():
        raise ValueError(f"Error: file does not exist: {file_path}")
    if file_path.is_dir():
        raise ValueError(f"{file_path} is a directory. Pass a specific file path.")
    if not file_path.is_file():
        raise ValueError(f"Path is not a regular file: {file_path}")

    return file_path.read_text(encoding="utf-8")


### `write_file`
The write_file function replaces the content of an existing file with new content. In this example, it only allows editing existing files.

The `write_text` method replaces the old content; it does not append new content. The system prompt instructs the agent to edit only the file requested by the user.

In [3]:
@tool
def write_file(file_input: str, content: str) -> str:
    """Replace the contents of an existing text file with the supplied content."""
    file_path = Path(file_input)

    if not file_path.exists():
        raise ValueError(f"File does not exist: {file_path}")
    if not file_path.is_file():
        raise ValueError(f"Path is not a regular file: {file_path}")

    file_path.write_text(content, encoding="utf-8")
    return f"File updated: {file_path}"


### `run_code`

The run_code function executes a Python script in a separate process.

- `sys.executable`: Uses the same Python interpreter as the current environment.
- `subprocess.run`: Executes the Python script in a separate process.
- `capture_output=True`: Captures the program's standard output and error messages.
- `timeout=20`:: Stops waiting if the script runs for more than 20 seconds.
- `--test`: `lc_tools.py` in a special test mode to check whether the agent can be built successfully.
**Safety:** Running arbitrary Python code can be risky. Do not execute untrusted scripts.

In [4]:
@tool
def run_code(script_path: str) -> str:
    """Execute a Python script and return its exit code, stdout, and stderr."""
    file_path = Path(script_path)

    if not file_path.exists():
        raise ValueError(f"Script does not exist: {file_path}")
    if not file_path.is_file() or file_path.suffix.lower() != ".py":
        raise ValueError("Please provide an existing .py file.")

    command = [sys.executable, str(file_path)]
    if file_path.name == "lc_tools.py":
        command.append("--test")

    try:
        result = subprocess.run(
            command,
            capture_output=True,
            text=True,
            timeout=20,
            check=False,
        )
    except subprocess.TimeoutExpired:
        return "Execution timed out after 20 seconds."

    return (
        f"Exit code: {result.returncode}\n"
        f"Output:\n{result.stdout}\n"
        f"Errors:\n{result.stderr}"
    )


## 4. System prompt

A system prompt defines the agent's **role, constraints, workflow, and safety rules**.

It is not a Python function and it does not execute actions by itself. It guides the model's decisions. Tools do the actual file reading/writing/running.

Important workflow:
1. Read the target file first.
2. Understand the requested change.
3. Edit only the requested file.
4. Run it.
5. Inspect real stdout/stderr and fix errors if appropriate.
6. Never claim success without execution evidence.

The file names below are examples for a project where `lc_tools.py` contains tools/model setup.


In [ ]:
SYSTEM_PROMPT = """
You are a self-editing Python coding agent.

Project files:
- lc_tools.py: Contains tools and model configuration.

Rules:
1. Always inspect the existing code before editing.
2. When modifying tools, read lc_tools.py first.
3. When modifying the graph, read lg_tools.py first.
4. Never pass "." or a directory to read_file.
5. Always provide the exact file path.
6. If the file path is unknown, ask the user.
7. Edit only the requested file.
8. After editing, run the modified code and inspect output and errors.
9. Fix errors and rerun when appropriate.
10. Never claim the code works unless it was executed and checked.
11. Do not make unrelated changes.
"""


## 5. Build the agent

### Chat model
`init_chat_model(...)` initializes the chat model. `model_provider="groq"` tells LangChain which provider integration to use. `api_key` authenticates. `temperature` controls output variability; lower values are often preferable for precise coding tasks.

### `create_agent`
It connects:
- model,
- tools,
- system prompt,
- optional checkpointer.

### `InMemorySaver`
Checkpointing means saving agent state between steps/turns. `InMemorySaver()` stores it in RAM. `thread_id` identifies a conversation thread; reusing the same ID can resume its saved state while the process is alive. It is not durable storage and is lost when the Python process/kernel restarts.


In [7]:
def build_agent():
    model = init_chat_model(
        "openai/gpt-oss-120b",
        model_provider="groq",
        api_key=api_key,
        temperature=0.2,
    )

    checkpointer = InMemorySaver()

    agent = create_agent(
        model=model,
        # tools=[read_file, write_file, run_code, addition, subtraction],
        tools=[read_file, write_file, run_code],
        system_prompt=SYSTEM_PROMPT,
        checkpointer=checkpointer,
    )
    return agent


agent = build_agent()
print("Agent built successfully.")


Agent built successfully.


## 6. Run a request

`agent.invoke(...)` sends a message to the agent. The agent may call tools multiple times before returning its final message.

`configurable.thread_id` identifies the checkpoint thread. Use the same ID to continue the same conversation; use a different ID for a fresh thread.

**Start with a harmless request** such as asking the agent to inspect a file. Then try a small, reversible edit.


In [ ]:
config = {"configurable": {"thread_id": "lc-demo-1"}}

user_text = input("Enter your request: ")

result = agent.invoke(
    {"messages": [{"role": "user", "content": user_text}]},
    config=config,
)

print(result["messages"][-1].content)


: 

## 8. Core concepts — quick recap

| Component | Role |
|---|---|
| Chat model | Understands user request and chooses whether/how to call tools |
| Tool | Callable Python capability exposed to the model |
| Tool docstring | Describes tool purpose and expected inputs to the model |
| System prompt | Sets agent role, rules, and workflow |
| `create_agent` | Connects model + tools + prompt into an agent |
| `InMemorySaver` | Saves checkpoint state in RAM |
| `thread_id` | Identifies a conversation/checkpoint thread |
| `invoke` | Starts an agent run and returns messages/result |
| `subprocess` | Executes a Python script outside the current process |


